### Ejercicio 01

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
datos_productos = [
(1, "Laptop Pro 15", "Laptops", 1299.99, 45),
(2, "Mouse Inalámbrico MX", "Accesorios", 59.99, 200),
(3, "Monitor UltraWide 34", "Monitores", 549.99, 30),
(4, "Teclado Mecánico RGB", "Accesorios", 129.99, 150),
(5, "Audífonos Noise Cancel", "Audio", 299.99, 75),
(6, "Webcam 4K Pro", "Accesorios", 89.99, 120),
(7, "Tablet Air 11", "Tablets", 799.99, 60),
(8, "Disco SSD 1TB", "Almacenamiento", 109.99, 300),
(9, "Laptop Gamer X", "Laptops", 1899.99, 25),
(10, "Parlante Bluetooth", "Audio", 79.99, 180),
]
schema_productos = StructType([
StructField("producto_id", IntegerType(), False),
StructField("nombre", StringType(), False),
StructField("categoria", StringType(), False),
StructField("precio", DoubleType(), False),
StructField("stock", IntegerType(), False),
])
df_productos = spark.createDataFrame(datos_productos, schema=schema_productos)
display(df_productos)

In [0]:
from pyspark.sql.types import DateType
from datetime import date
datos_ventas = [
(101, 1, 2, date(2026, 3, 1), "Ana García"),
(102, 3, 1, date(2026, 3, 1), "Carlos López"),
(103, 2, 3, date(2026, 3, 2), "María Rodríguez"),
(104, 5, 1, date(2026, 3, 2), "José Martínez"),
(105, 1, 1, date(2026, 3, 3), "Laura Sánchez"),
(106, 8, 2, date(2026, 3, 3), "Pedro Ramírez"),
(107, 4, 1, date(2026, 3, 4), "Sofía Torres"),
(108, 7, 1, date(2026, 3, 5), "Diego Hernández"),
(109, 2, 5, date(2026, 3, 5), "Valentina Cruz"),
(110, 10, 2, date(2026, 3, 6), "Andrés Morales"),
(111, 1, 1, date(2026, 3, 7), "Camila Flores"),
(112, 6, 3, date(2026, 3, 7), "Roberto Díaz"),
]
schema_ventas = StructType([
StructField("venta_id", IntegerType(), False),
StructField("producto_id", IntegerType(), False),
StructField("cantidad", IntegerType(), False),
StructField("fecha", DateType(), False),
StructField("cliente", StringType(), False),
])
df_ventas = spark.createDataFrame(datos_ventas, schema=schema_ventas)
display(df_ventas)

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
df_productos.write.format("delta").mode("overwrite").saveAsTable("workspace.default.productos")
df_ventas.write.format("delta").mode("overwrite").saveAsTable("workspace.default.ventas")

In [0]:
%sql
SHOW TABLES IN workspace.default;

In [0]:
%sql
DESCRIBE DETAIL workspace.default.productos;

In [0]:
%sql
DESCRIBE HISTORY workspace.default.productos;

In [0]:
%sql
DESCRIBE HISTORY workspace.default.productos;

### Ejercicio 02

In [0]:
%sql
UPDATE workspace.default.productos
SET precio = precio * 1.10
WHERE categoria = 'Laptops';

UPDATE workspace.default.productos
SET precio = precio * 0.85
WHERE categoria = 'Accesorios';

In [0]:
%sql
SELECT nombre, categoria, precio
FROM workspace.default.productos
ORDER BY categoria, nombre;

In [0]:
%sql
DESCRIBE HISTORY workspace.default.productos;

### Ejercicio 03

In [0]:
%sql
DELETE FROM workspace.default.productos
WHERE producto_id IN (6, 10);

In [0]:
%sql
SELECT * FROM workspace.default.productos ORDER BY producto_id;

SELECT COUNT(*) AS total_productos FROM workspace.default.productos;

In [0]:
%sql
INSERT INTO workspace.default.ventas VALUES
(113, 1, 1, '2026-03-08', 'Fernando Ruiz'),
(114, 4, 2, '2026-03-08', 'Isabella Vargas'),
(115, 5, 1, '2026-03-09', 'Mateo Castillo'),
(116, 7, 1, '2026-03-09', 'Luciana Peña'),
(117, 9, 1, '2026-03-10', 'Sebastián Ortega');

SELECT COUNT(*) AS total_ventas FROM workspace.default.ventas;

In [0]:
from datetime import date

nuevas_ventas = [
    (118, 3, 1, date(2026, 3, 10), "Gabriela Mendoza"),
    (119, 8, 3, date(2026, 3, 11), "Nicolás Herrera"),
    (120, 1, 1, date(2026, 3, 11), "Paula Jiménez")
]
df_nuevas = spark.createDataFrame(nuevas_ventas, schema=schema_ventas)
df_nuevas.write.format("delta").mode("append").saveAsTable("workspace.default.ventas")

In [0]:
%sql
SELECT COUNT(*) AS total_ventas FROM workspace.default.ventas;

### Ejercicio 05

In [0]:
datos_actualizacion = [
    (1, "Laptop Pro 15", "Laptops", 1399.99, 50),
    (3, "Monitor UltraWide 34", "Monitores", 499.99, 40),
    (11, "Cargador USB-C Rápido", "Accesorios", 34.99, 500),
    (12, "Smartwatch Deportivo", "Wearables", 249.99, 80)
]
df_actualizacion = spark.createDataFrame(datos_actualizacion, schema=schema_productos)
display(df_actualizacion)

df_actualizacion.createOrReplaceTempView("actualizacion_proveedor")

In [0]:
%sql
MERGE INTO workspace.default.productos AS target
USING actualizacion_proveedor AS source
ON target.producto_id = source.producto_id
WHEN MATCHED THEN
    UPDATE SET
    target.precio = source.precio,
    target.stock = source.stock
WHEN NOT MATCHED THEN
    INSERT (producto_id, nombre, categoria, precio, stock)
    VALUES (source.producto_id, source.nombre, source.categoria, source.precio, source.stock);

In [0]:
%sql
SELECT * FROM workspace.default.productos ORDER BY producto_id;

DESCRIBE HISTORY workspace.default.productos;

Reflexiones

Reflexión Ejercicio 1 (Exploración e Historial)

   Al ejecutar el comando DESCRIBE DETAIL, se observa que la tabla efectivamente tiene el formato Delta y se detalla la ruta exacta de los archivos. Posteriormente, mediante el DESCRIBE HISTORY, se nota que el sistema empezó a registrar la actividad desde el primer momento: la tabla inicia en la versión 0 con la primera operación de escritura. Es interesante ver cómo Delta rastrea todo desde el inicio.

Reflexión Ejercicio 2 (UPDATE y versiones)

  Al revisar el historial, ahora se muestran nuevas versiones generadas por las actualizaciones de precio. Se aprende que Delta Lake no modifica los archivos originales directamente. En su lugar, crea nuevos archivos con los datos actualizados y registra el cambio en el log. Esta característica es la que permite consultar versiones anteriores más adelante.

Reflexión Ejercicio 3 (DELETE)

  Aunque se utilizó el comando DELETE y los productos descontinuados ya no aparecen en el SELECT, se comprende que los datos no se perdieron de forma definitiva. Delta Lake simplemente los marca como inactivos en la versión actual, pero la información sigue guardada en el historial. Esto resulta ser una medida de seguridad muy útil en caso de que se necesite revertir un borrado accidental.

Reflexión Ejercicio 4 (INSERT)

  Al agregar las ventas mediante PySpark, se observa que fue clave utilizar el modo append. Esto garantiza que los registros nuevos se agreguen al final de la tabla sin afectar la información existente. Se deduce que si se hubiera aplicado el modo overwrite por error, se habría borrado todo el historial de ventas anterior, dejando únicamente los 3 registros nuevos.

Reflexión Ejercicio 5 (MERGE)

  El uso del comando MERGE resulta ser una herramienta muy eficiente. Se nota que permite realizar dos acciones en un solo paso: actualiza los precios y el stock de los productos existentes (condición MATCHED) y, simultáneamente, inserta los productos que son totalmente nuevos (condición NOT MATCHED). En las métricas del log se puede verificar el número exacto de filas afectadas por cada acción.